# OM_BKK_DATA Historical Weather Analysis

Exploratory analysis for the PostgreSQL table `"OM_BKK_DATA"`, loaded from the Open-Meteo Historical Weather API using the coordinates in `"Bangkok_Grid_9km"`.

Main goals:
- Check table coverage and missing values
- Understand rainfall patterns by hour, month, and grid point
- Compare weather variables against precipitation
- Create reusable daily/monthly summary datasets for modelling

## 1. Setup

Run the install cell only if your notebook kernel does not already have these packages.

In [ ]:
# Optional: uncomment if needed.
# %pip install pandas psycopg2-binary matplotlib seaborn numpy

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)

sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.figsize"] = (12, 5)

## 2. Connect To PostgreSQL

Defaults mirror the project scripts. Override with environment variables if needed.

In [ ]:
DB_CONFIG = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE", "postgres"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", "Pass1234"),
}

TABLE_NAME = '"OM_BKK_DATA"'
GRID_TABLE = '"Bangkok_Grid_9km"'

conn = psycopg2.connect(**DB_CONFIG)

def sql(query, params=None):
    return pd.read_sql_query(query, conn, params=params)

print("Connected")

## 3. Table Health

In [ ]:
summary = sql(f'''
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT grid_number) AS grid_count,
    MIN(local_forecast_time) AS min_time,
    MAX(local_forecast_time) AS max_time,
    COUNT(DISTINCT local_forecast_time) AS distinct_hours,
    COUNT(*)::numeric / NULLIF(COUNT(DISTINCT grid_number), 0) AS rows_per_grid
FROM {TABLE_NAME};
''')
summary

In [ ]:
columns = [
    "temperature_2m",
    "relative_humidity_2m",
    "pressure_msl",
    "surface_pressure",
    "dew_point_2m",
    "precipitation",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
]

missing_expr = ",\n    ".join(
    f"COUNT(*) FILTER (WHERE {column} IS NULL) AS missing_{column}"
    for column in columns
)
missing = sql(f"""
SELECT
    COUNT(*) AS row_count,
    {missing_expr}
FROM {TABLE_NAME};
""")
missing.T.rename(columns={0: "count"})

In [ ]:
coverage_by_year = sql(f'''
SELECT
    DATE_TRUNC('year', local_forecast_time)::date AS year,
    COUNT(*) AS rows,
    COUNT(DISTINCT grid_number) AS grids,
    COUNT(DISTINCT local_forecast_time) AS hours
FROM {TABLE_NAME}
GROUP BY 1
ORDER BY 1;
''')
coverage_by_year

## 4. Grid Overview

In [ ]:
grid = sql(f'''
SELECT grid_number, grid_row, grid_column, latitude, longitude
FROM {GRID_TABLE}
ORDER BY grid_number;
''')
grid.head(), grid.shape

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(grid["longitude"], grid["latitude"], c=grid["grid_number"], cmap="viridis", s=60)
for _, row in grid.iterrows():
    ax.text(row["longitude"], row["latitude"], str(int(row["grid_number"])), fontsize=7, ha="center", va="center", color="white")
ax.set_title("Bangkok 9 km Grid Points")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.colorbar(scatter, ax=ax, label="Grid number")
plt.show()

## 5. Daily Weather Summary

This aggregates hourly data into daily values by grid point. It is small enough to work with comfortably in memory.

In [ ]:
daily = sql(f'''
SELECT
    grid_number,
    grid_row,
    grid_column,
    latitude,
    longitude,
    local_forecast_time::date AS date,
    AVG(temperature_2m) AS temp_mean,
    MIN(temperature_2m) AS temp_min,
    MAX(temperature_2m) AS temp_max,
    AVG(relative_humidity_2m) AS humidity_mean,
    AVG(dew_point_2m) AS dew_point_mean,
    AVG(pressure_msl) AS pressure_msl_mean,
    AVG(surface_pressure) AS surface_pressure_mean,
    SUM(precipitation) AS precipitation_sum,
    MAX(precipitation) AS precipitation_hourly_max,
    COUNT(*) FILTER (WHERE precipitation >= 0.1) AS rain_hours,
    AVG(cloud_cover) AS cloud_cover_mean,
    AVG(wind_speed_10m) AS wind_speed_mean
FROM {TABLE_NAME}
GROUP BY 1,2,3,4,5,6
ORDER BY date, grid_number;
''')

daily["date"] = pd.to_datetime(daily["date"])
daily["month"] = daily["date"].dt.month
daily["year"] = daily["date"].dt.year
daily["is_rain_day"] = daily["precipitation_sum"] >= 0.1
daily.head()

In [ ]:
daily.describe(include="all").T

## 6. Rainfall Time Series

In [ ]:
city_daily = (
    daily.groupby("date", as_index=False)
    .agg(
        precipitation_mean=("precipitation_sum", "mean"),
        precipitation_max=("precipitation_sum", "max"),
        rain_grid_share=("is_rain_day", "mean"),
        temp_mean=("temp_mean", "mean"),
        humidity_mean=("humidity_mean", "mean"),
        cloud_cover_mean=("cloud_cover_mean", "mean"),
        wind_speed_mean=("wind_speed_mean", "mean"),
    )
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(city_daily["date"], city_daily["precipitation_mean"], linewidth=1, label="Mean daily precipitation")
ax.plot(city_daily["date"], city_daily["precipitation_max"], linewidth=0.8, alpha=0.6, label="Max grid daily precipitation")
ax.set_title("Bangkok Daily Precipitation Across Grid")
ax.set_ylabel("Precipitation (mm/day)")
ax.legend()
plt.show()

In [ ]:
top_rain_days = city_daily.sort_values("precipitation_max", ascending=False).head(20)
top_rain_days

## 7. Seasonality

In [ ]:
monthly = (
    daily.groupby(["year", "month"], as_index=False)
    .agg(
        precipitation_sum=("precipitation_sum", "sum"),
        precipitation_mean=("precipitation_sum", "mean"),
        rain_day_share=("is_rain_day", "mean"),
        temp_mean=("temp_mean", "mean"),
        humidity_mean=("humidity_mean", "mean"),
        cloud_cover_mean=("cloud_cover_mean", "mean"),
        wind_speed_mean=("wind_speed_mean", "mean"),
    )
)
monthly["month_name"] = pd.to_datetime(monthly["month"], format="%m").dt.month_name().str.slice(0, 3)
monthly.head()

In [ ]:
seasonal = monthly.groupby("month", as_index=False).agg(
    precipitation_sum_mean=("precipitation_sum", "mean"),
    rain_day_share_mean=("rain_day_share", "mean"),
    temp_mean=("temp_mean", "mean"),
    humidity_mean=("humidity_mean", "mean"),
)
seasonal["month_name"] = pd.to_datetime(seasonal["month"], format="%m").dt.month_name().str.slice(0, 3)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
sns.barplot(data=seasonal, x="month_name", y="precipitation_sum_mean", ax=axes[0], color="#4c78a8")
axes[0].set_title("Average Monthly Rainfall Across Available Years")
axes[0].set_ylabel("Total precipitation (mm)")
sns.lineplot(data=seasonal, x="month_name", y="rain_day_share_mean", marker="o", ax=axes[1], color="#f58518")
axes[1].set_ylabel("Share of grid-days with rain")
axes[1].set_xlabel("Month")
plt.show()

## 8. Hour Of Day Patterns

In [ ]:
hourly_pattern = sql(f'''
SELECT
    EXTRACT(HOUR FROM local_forecast_time)::int AS hour,
    AVG(temperature_2m) AS temp_mean,
    AVG(relative_humidity_2m) AS humidity_mean,
    AVG(cloud_cover) AS cloud_cover_mean,
    AVG(wind_speed_10m) AS wind_speed_mean,
    AVG(precipitation) AS precipitation_mean,
    AVG((precipitation >= 0.1)::int) AS rain_probability
FROM {TABLE_NAME}
GROUP BY 1
ORDER BY 1;
''')

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
sns.lineplot(data=hourly_pattern, x="hour", y="rain_probability", marker="o", ax=axes[0, 0])
axes[0, 0].set_title("Rain Probability By Hour")
sns.lineplot(data=hourly_pattern, x="hour", y="precipitation_mean", marker="o", ax=axes[0, 1])
axes[0, 1].set_title("Mean Precipitation By Hour")
sns.lineplot(data=hourly_pattern, x="hour", y="temp_mean", marker="o", ax=axes[1, 0])
axes[1, 0].set_title("Temperature By Hour")
sns.lineplot(data=hourly_pattern, x="hour", y="humidity_mean", marker="o", ax=axes[1, 1])
axes[1, 1].set_title("Humidity By Hour")
for ax in axes.flat:
    ax.set_xlabel("Hour")
    ax.set_xticks(range(0, 24, 3))
plt.tight_layout()
plt.show()

hourly_pattern

## 9. Spatial Rainfall Differences

In [ ]:
grid_rain = daily.groupby(
    ["grid_number", "grid_row", "grid_column", "latitude", "longitude"],
    as_index=False,
).agg(
    precipitation_daily_mean=("precipitation_sum", "mean"),
    precipitation_daily_total=("precipitation_sum", "sum"),
    rain_day_share=("is_rain_day", "mean"),
    hourly_max=("precipitation_hourly_max", "max"),
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc1 = axes[0].scatter(grid_rain["longitude"], grid_rain["latitude"], c=grid_rain["precipitation_daily_mean"], cmap="Blues", s=90)
axes[0].set_title("Mean Daily Precipitation By Grid")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
fig.colorbar(sc1, ax=axes[0], label="mm/day")

sc2 = axes[1].scatter(grid_rain["longitude"], grid_rain["latitude"], c=grid_rain["rain_day_share"], cmap="YlGnBu", s=90)
axes[1].set_title("Rain-Day Share By Grid")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
fig.colorbar(sc2, ax=axes[1], label="Share")
plt.show()

grid_rain.sort_values("precipitation_daily_mean", ascending=False).head(10)

## 10. Correlations With Rain

In [ ]:
corr_columns = [
    "temp_mean",
    "temp_min",
    "temp_max",
    "humidity_mean",
    "dew_point_mean",
    "pressure_msl_mean",
    "surface_pressure_mean",
    "cloud_cover_mean",
    "wind_speed_mean",
    "rain_hours",
    "precipitation_sum",
]

corr = daily[corr_columns].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", linewidths=0.5, ax=ax)
ax.set_title("Daily Feature Correlations")
plt.show()

corr["precipitation_sum"].sort_values(ascending=False)

## 11. Rain Thresholds

Use this to choose a rainfall threshold for classification targets.

In [ ]:
thresholds = [0.1, 0.5, 1, 2, 5, 10, 20]
threshold_summary = pd.DataFrame({
    "threshold_mm_day": thresholds,
    "grid_day_share": [(daily["precipitation_sum"] >= threshold).mean() for threshold in thresholds],
    "city_day_share_any_grid": [
        (daily.assign(hit=daily["precipitation_sum"] >= threshold).groupby("date")["hit"].max()).mean()
        for threshold in thresholds
    ],
})
threshold_summary

## 12. Save Analysis Extracts

Optional exports for modelling or charting outside the notebook.

In [ ]:
OUTPUT_DIR = "om_bkk_analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

daily.to_csv(os.path.join(OUTPUT_DIR, "om_bkk_daily_by_grid.csv"), index=False)
city_daily.to_csv(os.path.join(OUTPUT_DIR, "om_bkk_city_daily.csv"), index=False)
monthly.to_csv(os.path.join(OUTPUT_DIR, "om_bkk_monthly.csv"), index=False)
grid_rain.to_csv(os.path.join(OUTPUT_DIR, "om_bkk_grid_rain_summary.csv"), index=False)

print(f"Saved outputs to {OUTPUT_DIR}")

In [ ]:
conn.close()
print("Connection closed")